# 1. 기존 코드 구조 가져오기

In [1]:
from pathlib import Path
import sys, inspect, json, warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from xgboost import XGBClassifier

# 실행 위치가 저장소 루트 또는 그 하위인 경우 자동으로 src를 찾습니다.
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / 'src' / 'config.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f'저장소 루트(src/config.py)를 찾을 수 없습니다. 실행 위치: {start}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.validation import get_folds, run_cv
from src.evaluation import macro_f1, per_class_f1, log_result, save_predictions
import src.features as features

set_seed()
print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)
print('XGBoost:', __import__('xgboost').__version__)
# GPU 자동 탐지: 실제 XGBoost CUDA 학습을 소량 실행해 확인합니다.
def select_xgb_device():
    try:
        probe = XGBClassifier(
            n_estimators=1, max_depth=1, tree_method='hist',
            device='cuda', objective='binary:logistic',
            eval_metric='logloss', n_jobs=1, verbosity=0
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            probe.fit(np.array([[0.], [1.], [2.], [3.]], dtype=np.float32),
                      np.array([0, 0, 1, 1]))
        config = json.loads(probe.get_booster().save_config())
        actual = config.get('learner', {}).get('generic_param', {}).get('device', '')
        if str(actual).startswith('cuda'):
            print(f'XGBoost GPU 확인: {actual}')
            return 'cuda'
        print(f'XGBoost가 GPU 대신 {actual or "CPU"}로 실행되어 CPU를 사용합니다.')
    except Exception as exc:
        print(f'XGBoost GPU 확인 실패 ({type(exc).__name__}: {exc}); CPU로 전환합니다.')
    return 'cpu'

XGB_DEVICE = select_xgb_device()
print('선택한 XGBoost 장치:', XGB_DEVICE)

ROOT: /Users/admin/Documents/ah07-cancer-subtype
DATA_DIR: /Users/admin/Documents/ah07-cancer-subtype/data
XGBoost: 3.4.1
XGBoost가 GPU 대신 cpu로 실행되어 CPU를 사용합니다.
선택한 XGBoost 장치: cpu


In [2]:
# 최신 dev의 공통 피처 함수가 있는지 확인 (없으면 학습하지 않습니다).
required = ['mutation_count', 'mutation_type_counts', 'amino_acid_pair_counts', 'gene_truncation_flags']
missing = [name for name in required if not callable(getattr(features, name, None))]
if missing:
    raise RuntimeError('src/features.py에 최신 피처 함수가 없습니다: ' + ', '.join(missing) + '\n시윤님 PR 병합 후 최신 dev를 받아 주세요.')
for name in required:
    fn = getattr(features, name)
    print(name, inspect.signature(fn))

mutation_count (X: pandas.DataFrame) -> pandas.Series
mutation_type_counts (X)
amino_acid_pair_counts (X: pandas.DataFrame) -> pandas.DataFrame
gene_truncation_flags (X: pandas.DataFrame) -> pandas.DataFrame


In [3]:
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
X_test_raw = test[X_raw.columns].copy()
assert X_raw.columns.equals(X_test_raw.columns)
assert X_raw.index.equals(pd.RangeIndex(len(X_raw)))
assert X_test_raw.index.equals(pd.RangeIndex(len(X_test_raw)))
folds = get_folds(y, X=X_raw, classes=le.classes_)
assert len(folds) == 5
print('train:', X_raw.shape, 'test:', X_test_raw.shape, 'classes:', len(le.classes_))

train: (6201, 4384) test: (2546, 4384) classes: 26


In [4]:
# 함수는 팀 코드 그대로 호출합니다. 모든 피처는 행별로 계산해야 합니다.
def feature_frame(result, index, prefix):
    if isinstance(result, pd.Series):
        result = result.to_frame()
    if not isinstance(result, pd.DataFrame):
        raise TypeError(f'{prefix}: 반환값은 Series 또는 DataFrame이어야 합니다')
    if len(result) != len(index) or not result.index.equals(index):
        raise ValueError(f'{prefix}: 행 인덱스 또는 길이가 원본과 다릅니다')
    out = result.copy()
    out.columns = [f'{prefix}__{c}' for c in out.columns]
    return out

def make_E(X):
    binary = feature_frame(binarize_mutations(X), X.index, 'binary')
    count = feature_frame(np.log1p(features.mutation_count(X)), X.index, 'log_count')
    types = feature_frame(np.log1p(features.mutation_type_counts(X)), X.index, 'log_type')
    pairs = feature_frame(np.log1p(features.amino_acid_pair_counts(X)), X.index, 'log_pair')
    trunc = feature_frame(features.gene_truncation_flags(X), X.index, 'trunc')
    parts = [binary, count, types, pairs, trunc]
    if any(p.columns.has_duplicates for p in parts):
        raise ValueError('피처 내 중복 열 이름이 있습니다')
    result = pd.concat(parts, axis=1)
    if result.columns.has_duplicates or result.isna().any().any():
        raise ValueError('중복 열 또는 결측 피처가 있습니다')
    if not np.isfinite(result.to_numpy(dtype=np.float64)).all():
        raise ValueError('무한대/NaN 피처가 있습니다')
    return result.astype(np.float32)

X_E = make_E(X_raw)
X_test_E = make_E(X_test_raw)
assert X_E.columns.equals(X_test_E.columns), 'train/test 피처 열 또는 순서 불일치'
assert len(X_E) == len(y) and len(X_test_E) == len(test)
print('E 피처:', X_E.shape, X_test_E.shape)
print('참고: 기대 열 수 9174는 팀 최신 구현 기준이며, 실제 열 수를 확인하세요.')
if X_E.shape[1] != 9174:
    raise ValueError(f'팀 E 실험의 9174개 피처와 다릅니다: {X_E.shape[1]}개. 최신 함수의 열 정의를 확인하세요.')

E 피처: (6201, 9174) (2546, 9174)
참고: 기대 열 수 9174는 팀 최신 구현 기준이며, 실제 열 수를 확인하세요.


# 2. 신규 유전자 조합 피처 5개 생성

In [6]:
# STEP 2. 신규 유전자 조합 피처 생성

def make_subtype_features(X):
    genes = ["VHL", "TP53", "MET", "IDH1", "ATRX", "PTEN", "EGFR"]

    # 원본 유전자 데이터에서 변이 여부 추출
    mut = {
        g: (X[g].notna() & X[g].ne("WT")).astype("int8")
        for g in genes
    }

    features = pd.DataFrame(index=X.index)

    # 신장암 관련 피처 2개
    features["kidney_vhl0_tp53_1"] = (1 - mut["VHL"]) * mut["TP53"]
    features["kidney_vhl0_met_1"] = (1 - mut["VHL"]) * mut["MET"]

    # 뇌종양 관련 피처 3개
    features["brain_idh1_0_atrx_1"] = (1 - mut["IDH1"]) * mut["ATRX"]
    features["brain_idh1_0_pten_1"] = (1 - mut["IDH1"]) * mut["PTEN"]
    features["brain_egfr_0_pten_1"] = (1 - mut["EGFR"]) * mut["PTEN"]

    return features


# Train/Test 원본 데이터에 적용
subtype_train = make_subtype_features(X_raw)
subtype_test = make_subtype_features(X_test_raw)

print("신규 피처 Train:", subtype_train.shape)
print("신규 피처 Test:", subtype_test.shape)

display(subtype_train.head())

신규 피처 Train: (6201, 5)
신규 피처 Test: (2546, 5)


,kidney_vhl0_tp53_1,kidney_vhl0_met_1,brain_idh1_0_atrx_1,brain_idh1_0_pten_1,brain_egfr_0_pten_1
0,0,0,0,0,0
1,1,0,1,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,1,0,0,1,1


In [7]:
# 기존 Feature E와 신규 피처 결합

# STEP 3. Baseline / A / B / C 데이터셋 구성

kidney_cols = [
    "kidney_vhl0_tp53_1",
    "kidney_vhl0_met_1",
]

brain_cols = [
    "brain_idh1_0_atrx_1",
    "brain_idh1_0_pten_1",
    "brain_egfr_0_pten_1",
]

# 인덱스 일치 확인 (잘못된 행 결합 방지)
assert X_E.index.equals(subtype_train.index)

# 기존 Feature E
X_baseline = X_E.copy()

# 실험 A: 신장암 피처 2개 추가
X_A = pd.concat(
    [X_E, subtype_train[kidney_cols]], axis=1
)

# 실험 B: 뇌종양 피처 3개 추가
X_B = pd.concat(
    [X_E, subtype_train[brain_cols]], axis=1
)

# 실험 C: 전체 피처 5개 추가
X_C = pd.concat(
    [X_E, subtype_train], axis=1
)

experiments = {
    "Baseline": X_baseline,
    "A_kidney": X_A,
    "B_brain": X_B,
    "C_all": X_C,
}

for name, X in experiments.items():
    print(f"{name}: {X.shape}")


Baseline: (6201, 9174)
A_kidney: (6201, 9176)
B_brain: (6201, 9177)
C_all: (6201, 9179)


# 3. Baseline 모델만 학습

In [8]:
# STEP 4. Baseline XGBoost 교차검증

common = dict(
    n_estimators=250,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=len(le.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    device=XGB_DEVICE,
    random_state=SEED,
    n_jobs=-1,
)

# 1차 실험은 max_depth=3으로 고정
model = XGBClassifier(max_depth=3, **common)

# 기존 팀 공통 Fold 사용
baseline_result = run_cv(
    model,
    X_baseline,
    y,
    folds=folds,
)

print("Baseline CV Mean Macro F1:", baseline_result["mean"])
print("Baseline CV Std:", baseline_result["std"])
print("Baseline OOF Macro F1:", macro_f1(y, baseline_result["oof"]))

fold 1: Macro F1 = 0.4780
fold 2: Macro F1 = 0.4920
fold 3: Macro F1 = 0.4647
fold 4: Macro F1 = 0.4937
fold 5: Macro F1 = 0.4878
CV Macro F1 = 0.4833 (std 0.0108)
Baseline CV Mean Macro F1: 0.4832537411818092
Baseline CV Std: 0.010755666099830585
Baseline OOF Macro F1: 0.4842416742244882


# 4. 신규 피처 추가 실험

In [9]:
# STEP 5. 신규 피처 추가 실험 A/B/C

results = {
    "Baseline": {
        "CV_Mean": baseline_result["mean"],
        "CV_Std": baseline_result["std"],
        "OOF_Macro_F1": macro_f1(y, baseline_result["oof"]),
    }
}

cv_results = {"Baseline": baseline_result}

for name in ["A_kidney", "B_brain", "C_all"]:
    print(f"\n{'=' * 50}")
    print(f"실험: {name}")
    print(f"{'=' * 50}")

    # 매 실험마다 동일한 설정의 새 모델 생성
    model = XGBClassifier(max_depth=3, **common)

    result = run_cv(
        model,
        experiments[name],
        y,
        folds=folds,
    )

    cv_results[name] = result

    results[name] = {
        "CV_Mean": result["mean"],
        "CV_Std": result["std"],
        "OOF_Macro_F1": macro_f1(y, result["oof"]),
    }

# 결과 비교
results_df = pd.DataFrame(results).T

results_df["Delta_vs_Baseline"] = (
    results_df["CV_Mean"] - results_df.loc["Baseline", "CV_Mean"]
)

display(results_df.round(4))


실험: A_kidney
fold 1: Macro F1 = 0.4727
fold 2: Macro F1 = 0.4921
fold 3: Macro F1 = 0.4621
fold 4: Macro F1 = 0.4997
fold 5: Macro F1 = 0.4890
CV Macro F1 = 0.4831 (std 0.0137)

실험: B_brain
fold 1: Macro F1 = 0.4754
fold 2: Macro F1 = 0.4970
fold 3: Macro F1 = 0.4606
fold 4: Macro F1 = 0.5028
fold 5: Macro F1 = 0.4857
CV Macro F1 = 0.4843 (std 0.0151)

실험: C_all
fold 1: Macro F1 = 0.4821
fold 2: Macro F1 = 0.5021
fold 3: Macro F1 = 0.4622
fold 4: Macro F1 = 0.4938
fold 5: Macro F1 = 0.4950
CV Macro F1 = 0.4870 (std 0.0140)


,CV_Mean,CV_Std,OOF_Macro_F1,Delta_vs_Baseline
Baseline,0.4833,0.0108,0.4842,0.0000
A_kidney,0.4831,0.0137,0.4846,-0.0001
B_brain,0.4843,0.0151,0.4857,0.0011
C_all,0.4870,0.0140,0.4885,0.0038


- A (신장암 2개): 전체 성능 개선이 관찰되지 않음.
- B (뇌종양 3개): 소폭 개선됐지만 차이가 작음.
- C (전체 5개): 가장 높은 점수를 기록했으며, Baseline 대비 약 0.0038점 상승함.
- A 단독으로는 개선되지 않았는데, B와 함께 적용했을 때 가장 좋은 결과가 나왔음.
- 다만 이것만으로 피처 간 상호작용이 있다고 결론 내릴 수는 없으며, 트리 모델의 분할 구조가 달라지면서 발생한 차이일 수도 있음.

In [10]:
# 암종별 F1·Recall 비교

from sklearn.metrics import f1_score, recall_score

target_classes = ["KIPAN", "KIRC", "GBMLGG", "LGG"]

# 클래스 이름을 숫자 라벨로 변환
target_ids = le.transform(target_classes)

rows = []

for name, result in cv_results.items():
    y_pred = result["oof"]

    f1_values = f1_score(
        y, y_pred,
        labels=target_ids,
        average=None,
        zero_division=0
    )

    recall_values = recall_score(
        y, y_pred,
        labels=target_ids,
        average=None,
        zero_division=0
    )

    for cls, f1, recall in zip(
        target_classes, f1_values, recall_values
    ):
        rows.append({
            "Experiment": name,
            "Class": cls,
            "F1": f1,
            "Recall": recall
        })

class_results = pd.DataFrame(rows)

display(
    class_results.pivot(
        index="Class",
        columns="Experiment",
        values=["F1", "Recall"]
    ).round(4)
)

F1                            Recall                         
Experiment A_kidney B_brain Baseline   C_all A_kidney B_brain Baseline   C_all
Class                                                                         
GBMLGG       0.4939  0.4815   0.4855  0.4974   0.5249  0.5076   0.5076  0.5141
KIPAN        0.4211  0.4254   0.4152  0.4301   0.5204  0.5204   0.5107  0.5320
KIRC         0.2925  0.2896   0.2957  0.2970   0.2216  0.2216   0.2275  0.2246
LGG          0.4795  0.4878   0.4910  0.4842   0.4585  0.4803   0.4760  0.4672

#### Baseline vs C_all 비교
- KIPAN: 가장 뚜렷하게 개선됨. (F1과 Recall 모두 상승)
- GBMLGG: F1이 0.4855 → 0.4974로 개선됨.
- KIRC: F1은 거의 변화 없고 Recall은 소폭 하락함.
- LGG: F1과 Recall 모두 하락함.

#### 실험별로 비교
- 뇌종양 피처 B가 뇌종양 성능을 개선하지 못함. B_brain은 GBMLGG와 LGG의 F1이 모두 Baseline보다 낮아진 반면 KIPAN은 개선됨. 따라서 피처 이름이 특정 암종과 관련 있다고 해서 그 암종의 성능이 반드시 개선되는 건 아니라는 점을 보여줌.
- 전체 조합 C가 가장 좋지만 개선 폭은 작음. 전체 CV Macro F1은 0.4833 → 0.4870으로 약 0.0038 상승함. Fold별 점수는 일부 상승하고 일부 하락했기 때문에 일반화 성능이 확실히 개선됐다고 보기는 어려움

# 5. Baseline과 C_all의 오분류 변화 분석

In [11]:
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

# 기존 OOF 예측
pred_base = cv_results["Baseline"]["oof"]
pred_c = cv_results["C_all"]["oof"]

# 1. 전체 26개 암종별 F1 변화
class_ids = list(range(len(le.classes_)))

f1_base = f1_score(y, pred_base, labels=class_ids,
                   average=None, zero_division=0)
f1_c = f1_score(y, pred_c, labels=class_ids,
                average=None, zero_division=0)

f1_compare = pd.DataFrame({
    "Class": le.classes_,
    "Baseline_F1": f1_base,
    "C_all_F1": f1_c,
})

f1_compare["Delta_F1"] = (
    f1_compare["C_all_F1"] - f1_compare["Baseline_F1"]
)

print("=== 전체 26개 암종 F1 변화 ===")
display(f1_compare.sort_values("Delta_F1", ascending=False).round(4))

# 2. 주요 암종 간 오분류 방향
pairs = [
    ("KIPAN", "KIRC"),
    ("GBMLGG", "LGG"),
]

rows = []

for true_class, pred_class in pairs:
    for src, dst in [(true_class, pred_class),
                     (pred_class, true_class)]:
        true_id = le.transform([src])[0]
        pred_id = le.transform([dst])[0]

        mask = (y == true_id)

        n_base = ((pred_base == pred_id) & mask).sum()
        n_c = ((pred_c == pred_id) & mask).sum()

        rows.append({
            "True": src,
            "Predicted_as": dst,
            "Baseline": n_base,
            "C_all": n_c,
            "Change": n_c - n_base,
        })

print("=== 주요 암종 간 오분류 변화 ===")
display(pd.DataFrame(rows))

# 3. 개별 샘플 예측 개선/악화
base_correct = pred_base == y
c_correct = pred_c == y

print("=== 예측 변화 ===")
print("Baseline 오답 → C_all 정답:",
      ((~base_correct) & c_correct).sum())
print("Baseline 정답 → C_all 오답:",
      (base_correct & (~c_correct)).sum())
print("두 모델 모두 정답:",
      (base_correct & c_correct).sum())
print("두 모델 모두 오답:",
      ((~base_correct) & (~c_correct)).sum())

=== 전체 26개 암종 F1 변화 ===


,Class,Baseline_F1,C_all_F1,Delta_F1
5,DLBC,0.4167,0.4490,0.0323
1,BLCA,0.5114,0.5434,0.0320
19,SARC,0.1439,0.1691,0.0252
22,TGCT,0.5688,0.5856,0.0168
24,THYM,0.3197,0.3362,0.0165
8,KIPAN,0.4152,0.4301,0.0150
7,HNSC,0.3584,0.3727,0.0143
6,GBMLGG,0.4855,0.4974,0.0119
10,LAML,0.4983,0.5101,0.0117
18,PRAD,0.3471,0.3561,0.0090


=== 주요 암종 간 오분류 변화 ===


,True,Predicted_as,Baseline,C_all,Change
0,KIPAN,KIRC,87,81,-6
1,KIRC,KIPAN,173,176,3
2,GBMLGG,LGG,103,101,-2
3,LGG,GBMLGG,96,98,2


=== 예측 변화 ===
Baseline 오답 → C_all 정답: 134
Baseline 정답 → C_all 오답: 115
두 모델 모두 정답: 2937
두 모델 모두 오답: 3015


- C_all은 전체 성능을 소폭 개선했지만, KIPAN–KIRC와 GBMLGG–LGG의 라벨 혼동을 실질적으로 해결하지는 못함.